In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
import sunpy.visualization.colormaps as cm

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def show(data):
    x_lims = mdates.date2num([datetime(2024,1,1), datetime(2026,1,1)])
    y_lims = [-1, 1]

    fig, ax = plt.subplots(figsize=(14,9))
    ax.imshow(data, cmap='seismic', vmin=-10, vmax=10, interpolation='nearest',
              extent=(x_lims[0], x_lims[1], y_lims[0], y_lims[1]), aspect='auto', origin='lower')

    ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [3]:
files_hmi = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

data_hmi = []
dates_hmi = []

for file in files_hmi:
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data_hmi.append(data)

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))
    dates_hmi.append(date)

data_hmi = np.array(data_hmi)
dates_hmi = np.array(dates_hmi)

flux_hmi = np.nanmean(data_hmi, axis=-1)
flux_hmi = (flux_hmi[:,1:] + flux_hmi[:,:-1]) / 2
flux_hmi = np.nan_to_num(flux_hmi)

/tmp/ipykernel_47960/4212245791.py:20: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_hmi, axis=-1)


In [31]:
files_fdt = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))

data_fdt = []
weights = []
dates_fdt = []

for file in files_fdt:
    s = np.load(file)
    mean = s['mean']
    weight = s['weight']

    start = datetime.fromisoformat(str(s['start']))
    end = datetime.fromisoformat(str(s['end']))
    delta = end - start
    date = start + delta / 2

    data_fdt.append(mean)
    weights.append(weight)
    dates_fdt.append(date)

data_fdt = np.array(data_fdt)
weights = np.array(weights)
dates_fdt = np.array(dates_fdt)

flux_fdt = np.nanmean(data_fdt, axis=-1)
flux_fdt = np.nan_to_num(flux_fdt)
flux_fdt = (flux_fdt[:,1:] + flux_fdt[:,:-1]) / 2

weights = weights.clip(0,1)
data = np.nan_to_num(data_fdt * weights) + np.nan_to_num(data_hmi * (1 - np.nan_to_num(weights)))

flux = np.nanmean(data, axis=-1)
flux = np.nan_to_num(flux)
flux = (flux[:,1:] + flux[:,:-1]) / 2

/tmp/ipykernel_47960/3473191517.py:25: RuntimeWarning: Mean of empty slice
  flux_fdt = np.nanmean(data_fdt, axis=-1)


In [32]:
plt.figure(figsize=(14,8))
plt.imshow(weights[17])

In [5]:
dsine = 1 / 359.5
lati = np.arange(-1,1 + dsine / 2, dsine)
lati = np.arcsin(lati.clip(-1,1)) * 180 / np.pi
lat = (lati[1:] + lati[:-1]) / 2

In [6]:
show(rebin(flux_hmi, 8, axis=1).T)

In [13]:
show(rebin(flux_fdt, 8, axis=1).T)

In [14]:
show(rebin(flux, 8, axis=1).T)

In [39]:
R = 695e8
u = 1000
D = 500e10
lam = 45
dt = timedelta(days=1).total_seconds()

xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.5)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)


i0 = 4
y = flux_fdt[i0].copy()

Q = []
T = []

for t in np.arange(dates_fdt[i0], dates_fdt[-1], timedelta(days=1)):
    y_ = interp1d(flux, dates_fdt, t)
    y = advect(y, vi, dt, xi=xi, ai=li)
    y = diffuse(y, D, dt, xi=xi, ai=li)
    y[np.abs(lat) < lam] = y_[np.abs(lat) < lam]

    Q.append(y)
    T.append(t)

Q = np.array(Q)
T = np.array(T)

In [40]:
i = 17

plt.figure(figsize=(10,8))
plt.plot(lat, flux_fdt[i0])
plt.plot(lat, flux_fdt[i])
plt.plot(lat, interp1d(Q, T, dates_fdt[i]), '--')
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [29]:
show(Q.T)

In [11]:
(R * np.pi / 180) ** 2 / 2 / D / 24 / 3600


1.7029841341721634

In [12]:
1.7 / 27.27 * 360

22.442244224422442

In [14]:
R * np.pi / 180 / u

14.039396182130313